In [1]:
! nvidia-smi

Fri Mar 14 15:57:07 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 552.22                 Driver Version: 552.22         CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                     TCC/WDDM  | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A6000             WDDM  |   00000000:16:00.0 Off |                  Off |
| 30%   33C    P8             16W /  300W |    9093MiB /  49140MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from datasets import load_dataset
import torch
from transformers import LlavaNextForConditionalGeneration, LlavaNextProcessor
import gc
import time
from transformers import BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from trl import SFTConfig,SFTTrainer
from PIL import Image

In [3]:
#Code to login to huggingFace using write-token generated. This enables us to push the fine-tuned model/adapters to HuggingFace Hub

from huggingface_hub import HfFolder, whoami

#Replace 'your_token' with your actual Hugging Face token
hf_token = 'your_token' #'YOUR_HF_TOKEN'  #Write token for Llava-Mistral-7B Model
HfFolder.save_token(hf_token)

user = whoami()
print(f"Logged in as {user['name']}")

Logged in as aadhibest


In [4]:
#Set environment variable to a GPU number (either 0,1 or 2 based on availability) so as to limit the code to use only a single GPU and prevent multi-GPU errors
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [1]:
#Initialize System Message variable to be passed to dataset
system_message = """You are a Vision-Language Model specialized in interpreting PCB circuit designs and waveform simulation graphs. Your task is to analyze the provided image, determine whether it is a circuit schematic, PCB layout, or waveform simulation graph, and generate a structured, technically detailed description.

For circuit schematics and PCB layouts, identify the circuit topology, functional blocks, component details, signal flow, electrical parameters, and PCB layout considerations. For waveform simulation graphs, extract key waveform characteristics, analyze signal integrity, and assess circuit performance.

Focus on delivering precise, structured, and insightful technical descriptions based on visual information. Avoid unnecessary elaboration beyond what is essential for accuracy."""

In [6]:
#Convert Data tp 

def load_image(image_path, target_size=(512, 512)):
    """Cmpresses image to desired resolution

    Args:
        image_path (_type_): Path to image file
        target_size (tuple, optional): Specifies target resolution or size. Defaults to (512, 512).

    Returns:
        img.copy(): Copy of the compressed image file
    """
    with Image.open(image_path) as img:
        img = img.convert("RGB")  # Ensure compatibility
        img = img.resize(target_size, Image.LANCZOS) #High-Quality Downscaling
        return img.copy()

def format_data(sample):
    """Formats data to a desired template/format.
        This also adds system message to dataset.

    Args:
        sample (_type_): Sample/Row from dataset
    """
    return [
        {
            "role": "system",
            "content": [
                {
                    "type": "text",
                    "text": system_message
                }
            ],
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": load_image(sample["image"]),
                },
                {
                    "type": "text",
                    "text": sample["conversations"][0]["value"],
                }
            ],
        },
        {
            "role": "assistant",
            "content": [
                {
                    "type": "text",
                    "text": sample["conversations"][1]["value"]
                }
            ],
        },
    ]

In [2]:
#Load and split the dataset
raw_datasets = load_dataset("json", data_files="json_data_12_03_2025_final.json")
split_datasets = raw_datasets["train"].train_test_split(test_size=0.2)
train_dataset = split_datasets["train"] 
eval_dataset = split_datasets["test"]

print(f"Training Dataset Size: {len(train_dataset)}") 
print(f"Evaluation Dataset Size: {len(eval_dataset)}")

NameError: name 'load_dataset' is not defined

In [8]:
train_dataset[0]

{'id': '4086',
 'image': 'circuit_imgs_03/horizontal_flip/P2P_R150_L15_CHARIMP65_PROP0.2_TDR_ANALYSIS_horizontal_flip.png',
 'conversations': [{'from': 'human',
   'value': 'Describe this graph in detail.'},
  {'from': 'gpt',
   'value': 'The Time Domain Reflectometry (TDR) analysis of this point-to-point (P2P) topology circuit reveals a significant impedance discontinuity, resulting in notable reflections in voltage, current, and impedance profiles. The voltage response initially spikes sharply to about 250 mV within the first 0.2 ns, indicating a fast rise time. This is followed by a sudden secondary increase to nearly 400 mV around 2.2 ns, which points to an impedance mismatch or a change in characteristic impedance along the transmission line. The current waveform shows a similar pattern, reaching a steady-state value of approximately 4.5 mA before experiencing a sharp drop to about 2 mA at the 2.2 ns mark, confirming a reflection-induced change in current flow. The impedance profi

In [9]:
#Formate the dataset to desired format (also images are compressed)
train_dataset_formatted = [format_data(sample) for sample in train_dataset]
eval_dataset_formatted = [format_data(sample) for sample in eval_dataset]

In [10]:
train_dataset_formatted[0]

[{'role': 'system',
  'content': [{'type': 'text',
    'text': 'You are a Vision-Language Model specialized in interpreting PCB circuit designs and waveform simulation graphs. Your task is to analyze the provided image, determine whether it is a circuit schematic, PCB layout, or waveform simulation graph, and generate a structured, technically detailed description.\n\nFor circuit schematics and PCB layouts, identify the circuit topology, functional blocks, component details, signal flow, electrical parameters, and PCB layout considerations. For waveform simulation graphs, extract key waveform characteristics, analyze signal integrity, and assess circuit performance.\n\nFocus on delivering precise, structured, and insightful technical descriptions based on visual information. Avoid unnecessary elaboration beyond what is essential for accuracy.'}]},
 {'role': 'user',
  'content': [{'type': 'image',
    'image': <PIL.Image.Image image mode=RGB size=512x512>},
   {'type': 'text', 'text': '

In [11]:
# Save train_dataset and eval_dataset to disk
train_dataset_path = "./datasets/train_dataset_13_03_2025_llava_mistral"
eval_dataset_path = "./datasets/eval_dataset_13_03_2025_llava_mistral"

train_dataset.save_to_disk(train_dataset_path)
eval_dataset.save_to_disk(eval_dataset_path)

print(f"Train dataset saved to: {train_dataset_path}")
print(f"Eval dataset saved to: {eval_dataset_path}")

Saving the dataset (0/1 shards):   0%|          | 0/3585 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/897 [00:00<?, ? examples/s]

Train dataset saved to: ./datasets/train_dataset_13_03_2025_llava_mistral
Eval dataset saved to: ./datasets/eval_dataset_13_03_2025_llava_mistral


In [12]:
#Remove Model and Clean GPU

def clear_memory():
    """Clears variables from memory to clear the GPU memory
    """
    # Delete variables if they exist in the current global scope
    if 'inputs' in globals(): del globals()['inputs']
    if 'model' in globals(): del globals()['model']
    if 'processor' in globals(): del globals()['processor']
    if 'trainer' in globals(): del globals()['trainer']
    if 'peft_model' in globals(): del globals()['peft_model']
    if 'bnb_config' in globals(): del globals()['bnb_config']
    time.sleep(2)

    # Garbage collection and clearing CUDA memory
    gc.collect()
    time.sleep(2)
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    time.sleep(2)
    gc.collect()
    time.sleep(2)

    print(f"GPU allocated memory: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    print(f"GPU reserved memory: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

clear_memory() #Clearing variable, model and GPU

GPU allocated memory: 0.00 GB
GPU reserved memory: 0.00 GB


In [13]:
#Specify Model to be used
model_id = "llava-hf/llava-v1.6-mistral-7b-hf"

In [14]:
#Enable 4 bit quantization with Quantization config
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
)

#Load model
model = LlavaNextForConditionalGeneration.from_pretrained(model_id,
                                                      quantization_config=quantization_config,
                                                      torch_dtype=torch.float16)

#Intialize processor
processor = LlavaNextProcessor.from_pretrained(model_id)    

`low_cpu_mem_usage` was None, now default to True since model is quantized.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.48, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [15]:
# Configure LoRA
peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.05,
    r=8,
    bias="none",
    target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM",
)

# Apply PEFT model adaptation
peft_model = get_peft_model(model, peft_config)

# Print trainable parameters
peft_model.print_trainable_parameters()

trainable params: 4,194,304 || all params: 7,570,941,952 || trainable%: 0.0554


In [16]:
# # Configure LoRA
# peft_config = LoraConfig(
#     r=8,
#     lora_alpha=8,
#     lora_dropout=0.1,
#     target_modules=['down_proj','o_proj','k_proj','q_proj','gate_proj','up_proj','v_proj'],
#     use_dora=True,
#     init_lora_weights="gaussian"
# )

# # Apply PEFT model adaptation
# peft_model = get_peft_model(model, peft_config)

# # Print trainable parameters
# peft_model.print_trainable_parameters()

##  ----- Output : trainable params: 23,601,152 || all params: 7,590,348,800 || trainable%: 0.3109 -----

############## We chose previous lora config since we have less data, it is better to train lesser parameters for achieving our desired output ######################

In [17]:
#Specify training arguments including hyperparameters, output directory (to save the model), logging directory (to save the logs), logging method ('tensorborad' or 'wandb')
training_args = SFTConfig(
    output_dir="llava-1.6-mistral-7b-13-03-2025", 
    logging_dir="llava-1.6-mistral-7b-13-03-2025/logs",
    num_train_epochs=1,  
    per_device_train_batch_size=4,  
    gradient_accumulation_steps=4, 
    remove_unused_columns = False,
    optim="adamw_torch_fused",  
    learning_rate=2e-4,  
    lr_scheduler_type="constant",
    logging_steps=10, 
    save_strategy="steps", 
    save_steps=25,  
    save_total_limit=10,
    metric_for_best_model="eval_loss",  
    greater_is_better=False, 
    fp16=True,  
    max_grad_norm=0.3,  
    warmup_ratio=0.03, 
    push_to_hub=True, 
    report_to="tensorboard",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    dataset_text_field="",  
    dataset_kwargs={"skip_prepare_dataset": True},
)

In [18]:
def collate_fn(examples):
    """Function helps in collating the dataset by applying the chat template as per the model and also converts the data to batch variables that are passed to model as input

    Args:
        examples: Rows/Sampled from the formatted dataset

    Returns:
        batch: Variables that are passed as input to model
    """
    # Render the chat using the LLaVA-1.6-Mistral chat template
    texts = [processor.apply_chat_template(example, tokenize=False) for example in examples]

    # Process images: assume that in each example, the first image is located at:
    # example[1]['content'][0]['image']
    image_inputs = []
    for example in examples:
        # This structure is based on your previous models; adjust if necessary.
        image = example[1]['content'][0]['image']
        if image.mode != 'RGB':
            image = image.convert('RGB')
        image_inputs.append([image])

    # Use the processor to tokenize text and process images
    batch = processor(text=texts, images=image_inputs, return_tensors="pt", padding=True)
    input_ids = batch["input_ids"]
    labels = input_ids.clone()
    # Mask out the padding tokens
    labels[input_ids == processor.tokenizer.pad_token_id] = -100

    # Optionally, mask the <image> token if defined in the tokenizer's additional_special_tokens (i.e, we are masking the <image> token so that the model doesn't learn it as a text while training)
    try:
        image_token_id = processor.tokenizer.additional_special_tokens_ids[
            processor.tokenizer.additional_special_tokens.index("<image>")
        ]
        labels[input_ids == image_token_id] = -100
    except Exception:
        pass

    # Dynamic loss masking: only compute loss on the assistant's reply.
    # Here we assume that the user prompt ends with the closing tag "[/INST]".
    # Tokenize the closing marker as used in the chat template.
    inst_close_ids = processor.tokenizer(" [/INST]", add_special_tokens=False)["input_ids"]
    
    # For each sequence, find the last occurrence of the closing marker and mask everything before it.
    for i in range(labels.shape[0]):
        seq = input_ids[i].tolist()
        last_idx = -1
        # Slide over the sequence to match the marker
        for j in range(len(seq) - len(inst_close_ids) + 1):
            if seq[j:j+len(inst_close_ids)] == inst_close_ids:
                last_idx = j + len(inst_close_ids)
        # If found, mask all tokens before last_idx (they belong to system/user prompt)
        if last_idx > 0:
            labels[i, :last_idx] = -100

    batch["labels"] = labels
    return batch


In [19]:
#Specify the trainer with the arguments such as the model to be trained, training arguments, etc.
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset_formatted,
    eval_dataset=eval_dataset_formatted,
    data_collator=collate_fn,
    peft_config=peft_config,
    tokenizer=processor.tokenizer,
)

C:\Users\hal-pg2425-5\AppData\Local\Temp\8\ipykernel_72588\2552207196.py:1: FutureWarning: `tokenizer` is deprecated and removed starting from version 0.16.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  trainer = SFTTrainer(
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [20]:
#Launch a tensorboard session to log the data from the training/fine-tuning process to the logging directory
%load_ext tensorboard
%tensorboard --logdir llava-1.6-mistral-7b-13-03-2025/logs --port=6011

In [21]:
#Train/Fine-tune the model
trainer.train()

c:\Users\hal-pg2425-5\.conda\envs\multivlm02\Lib\site-packages\bitsandbytes\nn\modules.py:451: UserWarning: Input type into Linear4bit is torch.float16, but bnb_4bit_compute_dtype=torch.float32 (default). This will lead to slow inference or training speed.
  warnings.warn(
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
10,34.647900
20,21.742300
30,16.058700
40,15.009500
50,14.722500
60,14.590300
70,14.451000
80,14.449300
90,14.419200
100,14.302500


TrainOutput(global_step=224, training_loss=15.668747799737114, metrics={'train_runtime': 15767.5295, 'train_samples_per_second': 0.227, 'train_steps_per_second': 0.014, 'total_flos': 5.638013695464653e+17, 'train_loss': 15.668747799737114})

In [22]:
#Manually save the model to the output directory
trainer.save_model(training_args.output_dir)

In [23]:
#Push the model to HuggingFace Hub
trainer.push_to_hub()

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/aadhibest/llava-1.6-mistral-7b-13-03-2025/commit/92726a5b93e2364be24203a38b41d22e11bf150a', commit_message='End of training', commit_description='', oid='92726a5b93e2364be24203a38b41d22e11bf150a', pr_url=None, repo_url=RepoUrl('https://huggingface.co/aadhibest/llava-1.6-mistral-7b-13-03-2025', endpoint='https://huggingface.co', repo_type='model', repo_id='aadhibest/llava-1.6-mistral-7b-13-03-2025'), pr_revision=None, pr_num=None)

In [24]:
#Clear the variables freeing GPU memory
clear_memory()

GPU allocated memory: 0.02 GB
GPU reserved memory: 0.07 GB
